# 🔍 Qdrant Hybrid Search Demo

**Pipeline:** Dense (cosine) + Sparse (BM25/SPLADE) → RRF Fusion → Cross-Encoder Reranking

| Step | Model | Purpose |
|------|-------|---------|
| Dense embed | `BAAI/bge-small-en-v1.5` | Semantic cosine search |
| Sparse embed | `prithivida/Splade_PP_en_v1` | BM25-style keyword search |
| Fusion | RRF (Reciprocal Rank Fusion) | Merge both result lists |
| Rerank | `cross-encoder/ms-marco-MiniLM-L-6-v2` | Score query-doc pairs jointly |

## 📦 Install Dependencies

In [ ]:
%pip install qdrant-client fastembed sentence-transformers python-dotenv --quiet

## ⚙️ Configuration

All model and search settings are defined here — change them to swap models.

In [7]:
import os
from dotenv import load_dotenv

load_dotenv()

# ── Models ──────────────────────────────────────────────────────────
DENSE_MODEL  = "BAAI/bge-small-en-v1.5"              # cosine / dense
SPARSE_MODEL = "Qdrant/bm25" #"prithivida/Splade_PP_en_v1"           # BM25-style / sparse
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2" # cross-encoder reranker

# ── Collection ──────────────────────────────────────────────────────
COLLECTION   = "hybrid_demo"

# ── Search params ───────────────────────────────────────────────────
TOP_K_SEARCH = 20   # candidates retrieved before reranking
TOP_K_RERANK = 5    # final results after reranking

# ── Qdrant connection ───────────────────────────────────────────────
QDRANT_URL     = os.getenv("QDRANT_URL", "http://34.134.83.51:6333")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "08e9be25c9d4bbfd15ea69e99f36e2fc898d536e4daa2e3e3e2a931bf73269d4")  # None for local

print(f"Dense  model : {DENSE_MODEL}")
print(f"Sparse model : {SPARSE_MODEL}")
print(f"Rerank model : {RERANK_MODEL}")
print(f"Qdrant URL   : {QDRANT_URL}")

Dense  model : BAAI/bge-small-en-v1.5
Sparse model : Qdrant/bm25
Rerank model : cross-encoder/ms-marco-MiniLM-L-6-v2
Qdrant URL   : http://34.134.83.51:6333


## 📚 Sample Documents

In [8]:
DOCUMENTS = [
    "Qdrant is a vector database optimized for high-performance similarity search.",
    "Hybrid search combines sparse BM25 retrieval with dense vector embeddings.",
    "Cross-encoders rerank candidates by scoring query-document pairs jointly.",
    "RAG systems retrieve relevant chunks and pass them to a language model.",
    "SPLADE produces sparse vectors that mimic BM25 term-weighting behavior.",
    "BGE embeddings from BAAI are popular for dense semantic search tasks.",
    "Reranking improves precision by re-scoring the top-K retrieved candidates.",
    "Cosine similarity measures the angle between two dense embedding vectors.",
    "FastEmbed is a lightweight library for generating embeddings efficiently.",
    "LangChain and LlamaIndex both support Qdrant as a vector store backend.",
]

print(f"Loaded {len(DOCUMENTS)} documents")
for i, doc in enumerate(DOCUMENTS):
    print(f"  {i}: {doc}")

Loaded 10 documents
  0: Qdrant is a vector database optimized for high-performance similarity search.
  1: Hybrid search combines sparse BM25 retrieval with dense vector embeddings.
  2: Cross-encoders rerank candidates by scoring query-document pairs jointly.
  3: RAG systems retrieve relevant chunks and pass them to a language model.
  4: SPLADE produces sparse vectors that mimic BM25 term-weighting behavior.
  5: BGE embeddings from BAAI are popular for dense semantic search tasks.
  6: Reranking improves precision by re-scoring the top-K retrieved candidates.
  7: Cosine similarity measures the angle between two dense embedding vectors.
  8: FastEmbed is a lightweight library for generating embeddings efficiently.
  9: LangChain and LlamaIndex both support Qdrant as a vector store backend.


## 🤖 Load Models

In [9]:
from fastembed import TextEmbedding, SparseTextEmbedding
from sentence_transformers import CrossEncoder

print("Loading dense embedding model...")
dense_model = TextEmbedding(model_name=DENSE_MODEL)
print(f"  ✅ {DENSE_MODEL}")

print("Loading sparse embedding model...")
sparse_model = SparseTextEmbedding(model_name=SPARSE_MODEL)
print(f"  ✅ {SPARSE_MODEL}")

print("Loading cross-encoder reranker...")
rerank_model = CrossEncoder(RERANK_MODEL)
print(f"  ✅ {RERANK_MODEL}")

Loading dense embedding model...
  ✅ BAAI/bge-small-en-v1.5
Loading sparse embedding model...
  ✅ Qdrant/bm25
Loading cross-encoder reranker...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

  ✅ cross-encoder/ms-marco-MiniLM-L-6-v2


## 🔌 Connect to Qdrant

> **Run Qdrant locally:** `docker run -p 6333:6333 qdrant/qdrant`

In [10]:
from qdrant_client import QdrantClient

client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
)

# Verify connection
info = client.get_collections()
print(f"✅ Connected to Qdrant at {QDRANT_URL}")
print(f"   Existing collections: {[c.name for c in info.collections]}")

/tmp/ipykernel_65909/1917800879.py:3: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(


✅ Connected to Qdrant at http://34.134.83.51:6333
   Existing collections: []


## 📦 Create Collection

The collection holds **two vector types**:
- `dense` — fixed-size float vectors (cosine distance)
- `sparse` — variable-length sparse vectors (SPLADE/BM25)

In [11]:
from qdrant_client.models import (
    Distance, VectorParams,
    SparseVectorParams, SparseIndexParams,
)

# Detect dense vector dimension from model
sample_vec = list(dense_model.embed(["test"]))[0]
DENSE_DIM = len(sample_vec)
print(f"Dense vector dimension: {DENSE_DIM}")

if client.collection_exists(COLLECTION):
    print(f"⚠️  Collection '{COLLECTION}' already exists — deleting and recreating...")
    client.delete_collection(COLLECTION)

client.create_collection(
    collection_name=COLLECTION,
    vectors_config={
        "dense": VectorParams(
            size=DENSE_DIM,
            distance=Distance.COSINE,
        )
    },
    sparse_vectors_config={
        "sparse": SparseVectorParams(
            index=SparseIndexParams(on_disk=False)
        )
    },
)

print(f"✅ Collection '{COLLECTION}' created (dense_dim={DENSE_DIM})")

Dense vector dimension: 384
✅ Collection 'hybrid_demo' created (dense_dim=384)


## 📥 Index Documents

In [12]:
import uuid
from qdrant_client.models import PointStruct, SparseVector

print(f"Embedding {len(DOCUMENTS)} documents...")

dense_vecs  = list(dense_model.embed(DOCUMENTS))
sparse_vecs = list(sparse_model.embed(DOCUMENTS))

print(f"  Dense  vectors : {len(dense_vecs)} × {len(dense_vecs[0])}")
print(f"  Sparse vectors : {len(sparse_vecs)} (variable length)")

points = [
    PointStruct(
        id=str(uuid.uuid4()),
        vector={
            "dense": dv.tolist(),
            "sparse": SparseVector(
                indices=sv.indices.tolist(),
                values=sv.values.tolist(),
            ),
        },
        payload={"text": doc, "doc_id": i},
    )
    for i, (doc, dv, sv) in enumerate(zip(DOCUMENTS, dense_vecs, sparse_vecs))
]

client.upsert(collection_name=COLLECTION, points=points)
print(f"\n✅ Indexed {len(points)} points into '{COLLECTION}'")

Embedding 10 documents...
  Dense  vectors : 10 × 384
  Sparse vectors : 10 (variable length)

✅ Indexed 10 points into 'hybrid_demo'


In [17]:
sparse_q = list(sparse_model.embed(["Qdrant is a vector database"]))[0]

print(f"Non-zero elements : {len(sparse_q.indices)}")   # e.g. 8
print(f"Max index value   : {max(sparse_q.indices)}")   # e.g. 28,000+
print(f"Indices           : {sparse_q.indices}")
print(f"Values            : {sparse_q.values}")

Non-zero elements : 3
Max index value   : 1955147705
Indices           : [ 802574768 1955147705 1423958341]
Values            : [1.67868852 1.67868852 1.67868852]


## 🔍 Hybrid Search Function

Uses **Reciprocal Rank Fusion (RRF)** to merge dense and sparse result lists.

In [23]:
from qdrant_client.models import Prefetch, FusionQuery, Fusion

def hybrid_search(query: str, top_k: int = TOP_K_SEARCH) -> list:
    """Retrieve top_k candidates using BM25 + cosine hybrid search with RRF fusion."""
    # Embed query with both models
    dense_q  = list(dense_model.embed([query]))[0].tolist()
    sparse_q = list(sparse_model.embed([query]))[0]

    results = client.query_points(
        collection_name=COLLECTION,
        prefetch=[
            # Branch 1 — dense cosine similarity
            Prefetch(
                query=dense_q,
                using="dense",
                limit=top_k,
            ),
            # Branch 2 — sparse BM25-style (SPLADE)
            Prefetch(
                query=SparseVector(
                    indices=sparse_q.indices.tolist(),
                    values=sparse_q.values.tolist(),
                ),
                using="sparse",
                limit=top_k,
            ),
        ],
        # RRF merges both ranked lists into one
        query=FusionQuery(fusion=Fusion.RRF),
        limit=top_k,
        with_payload=True,
    )

    return results.points

print("✅ hybrid_search() defined")

✅ hybrid_search() defined


## 🎯 Reranking Function

The cross-encoder scores each (query, document) pair jointly — much more precise than bi-encoder similarity.

In [24]:
def rerank(query: str, candidates: list, top_k: int = TOP_K_RERANK) -> list:
    """Rerank candidates using the cross-encoder model."""
    if not candidates:
        return []

    texts  = [c.payload["text"] for c in candidates]
    pairs  = [[query, t] for t in texts]
    scores = rerank_model.predict(pairs)

    ranked = sorted(
        zip(candidates, scores),
        key=lambda x: x[1],
        reverse=True,
    )
    return ranked[:top_k]

print("✅ rerank() defined")

✅ rerank() defined


## 🚀 Full Search Pipeline

In [25]:
def search(query: str) -> list:
    """Full pipeline: hybrid retrieval → cross-encoder reranking."""
    print(f'\n🔍 Query: "{query}"')
    print("─" * 60)

    # Step 1 — hybrid retrieval
    candidates = hybrid_search(query)
    print(f"📋 Retrieved {len(candidates)} candidates (BM25 + cosine + RRF)")

    # Step 2 — rerank
    reranked = rerank(query, candidates)
    print(f"🎯 Top {len(reranked)} after reranking:\n")

    for rank, (point, score) in enumerate(reranked, 1):
        print(f"  {rank}. [score={score:.4f}]  {point.payload['text']}")

    return reranked

print("✅ search() pipeline defined")

✅ search() pipeline defined


## 🧪 Run Queries

In [26]:
results = search("how does hybrid search work?")


🔍 Query: "how does hybrid search work?"
────────────────────────────────────────────────────────────
📋 Retrieved 10 candidates (BM25 + cosine + RRF)
🎯 Top 5 after reranking:

  1. [score=5.9475]  Hybrid search combines sparse BM25 retrieval with dense vector embeddings.
  2. [score=-8.2540]  Qdrant is a vector database optimized for high-performance similarity search.
  3. [score=-9.8422]  BGE embeddings from BAAI are popular for dense semantic search tasks.
  4. [score=-9.9562]  Cross-encoders rerank candidates by scoring query-document pairs jointly.
  5. [score=-11.0826]  RAG systems retrieve relevant chunks and pass them to a language model.


In [ ]:
results = search("what is used for reranking results?")

In [ ]:
results = search("which embeddings are good for semantic search?")

## 🔬 Try Your Own Query

In [ ]:
my_query = "what databases support RAG pipelines?"

results = search(my_query)

## 🧹 Cleanup (optional)

In [ ]:
# Uncomment to delete the collection
# client.delete_collection(COLLECTION)
# print(f"🗑️  Collection '{COLLECTION}' deleted")

In [1]:
# test google cloud

In [5]:
from qdrant_client import QdrantClient
import os
from dotenv import load_dotenv

load_dotenv("/home/amir/.env")

True

In [8]:
os.getenv("QDRANT_URL")

'http://34.57.176.168:6333'

In [9]:
client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
)

collections = client.get_collections()
print("✅ Connected!")
print(f"Collections: {collections}")

/tmp/ipykernel_494380/2336825283.py:1: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(
/tmp/ipykernel_494380/2336825283.py:1: UserWarning: Qdrant client version 1.14.2 is incompatible with server version 1.18.1. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  client = QdrantClient(


✅ Connected!
Collections: collections=[]
